In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tabulate import tabulate
import matplotlib.pyplot as plt

import geopandas as gpd

# Investigating DLL data on urban areas

## data_final.gpkg

In [2]:
gpkg_path = Path("../data/input/DLL") / "data_final.gpkg"

current_dir = Path.cwd()
print(f"Current working directory: {current_dir}")

# List layers - requires pyogrio (the current geopandas default)
layers_info = gpd.list_layers(gpkg_path)
print("\nLayers in file:")
print(layers_info)

layer_name = layers_info["name"].iloc[0]
gdf = gpd.read_file(gpkg_path, layer=layer_name)

print("\nNumber of features:", len(gdf))
print("Columns:", gdf.columns.tolist())
print("Geometry type(s):", gdf.geom_type.unique())
print("CRS:", gdf.crs)
print("Bounding box:", gdf.total_bounds)
print(gdf.head())

Current working directory: k:\PythonWork\downscaling\Kaya_downscaling\downscaling

Layers in file:
         name geometry_type
0  data_final  MultiPolygon

Number of features: 356508
Columns: ['UID', 'NAME_1', 'NAME_2', 'NAME_3', 'NAME_4', 'NAME_5', 'GGMCF', 'EDGAR', 'DEGURBA_L1', 'DEGURBA_L2', 'Growth_Rate', 'GGMCF_2022', 'GID_0', 'NAME_0', 'GID_1', 'ENGTYPE_1', 'GID_2', 'ENGTYPE_2', 'GID_3', 'ENGTYPE_3', 'GID_4', 'ENGTYPE_4', 'GID_5', 'ENGTYPE_5', 'CONTINENT', 'GDAM_ID', 'POP', 'GDP', 'BUILT_SUM', 'BUILT_SUM_BASE', 'imp_change_area', 'imp_total_base', 'imp_total', 'ELEC_SUM', 'area', 'GDP_PC', 'POP_DENS', 'imp_prop', 'imp_prop_base', 'built_prop', 'built_prop_base', 'built_prop_absolute_trend', 'built_prop_relative_trend', 'built_prop_relative_trend_cut', 'country_area_sum', 'country_gdp_sum', 'country_pop_sum', 'country_built_sum', 'region_area_sum', 'region_gdp_sum', 'region_pop_sum', 'region_built_sum', 'area_prop_adm_0', 'GDP_prop_adm_0', 'POP_prop_adm_0', 'BUILT_prop_adm_0', 'ar

In [3]:
# print first row
GID_0s = ["NLD"]
df_selected = gdf[gdf["GID_0"].isin(GID_0s)].drop(columns="geometry")
df_selected_sample = df_selected.sample(n=5, random_state=42)
print(tabulate(df_selected_sample, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))

+---------+---------------+-----------------+----------+----------+----------+---------------+---------------+--------------+--------------+---------------+---------------+---------+-------------+----------+-------------+-------------+--------------+---------+-------------+---------+-------------+---------+-------------+-------------+-------------+----------+-----------------+--------------+------------------+-------------------+------------------+--------------+---------------+---------------+----------+------------+------------+-----------------+--------------+-------------------+-----------------------------+-----------------------------+---------------------------------+--------------------+-------------------+-------------------+---------------------+-------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+--------------------+-------------------+------------------+------------------+-----------------

## data_timeseries_70.csv

In [4]:
# read in csv file with time series data
csv_path = Path("../data/input/DLL") / "data_timeseries_70.csv"
df_time_series = pd.read_csv(csv_path, sep=",")
print(df_time_series.dtypes)
df_time_series[df_time_series["GDAM_id"].str.startswith("NLD")].sample(10, random_state=42)

GDAM_id          object
GID_1            object
cluster_2015      int64
cluster_2020    float64
cluster_2030    float64
cluster_2040    float64
cluster_2050    float64
cluster_2060    float64
cluster_2070    float64
cluster_2080    float64
cluster_2090    float64
cluster_2100    float64
dtype: object


,GDAM_id,GID_1,cluster_2015,cluster_2020,cluster_2030,cluster_2040,cluster_2050,cluster_2060,cluster_2070,cluster_2080,cluster_2090,cluster_2100
215635,NLD.9.48_1,NLD.9_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215450,NLD.4.7_1,NLD.4_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215664,NLD.10.21_1,NLD.10_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215590,NLD.8.59_1,NLD.8_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215464,NLD.4.29_1,NLD.4_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215740,NLD.14.53_2,NLD.14_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215423,NLD.2.4_1,NLD.2_1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
215465,NLD.4.30_1,NLD.4_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215548,NLD.8.12_1,NLD.8_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
215702,NLD.12.7_1,NLD.12_1,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [5]:
GID_2_selection = ["NLD.9.48_1"]
df1 = gdf[gdf["GID_2"].isin(GID_2_selection)]
df2 = df_time_series[df_time_series["GDAM_id"].isin(GID_2_selection)].copy()
print(tabulate(df1, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))
print(f"*"*100)
print(tabulate(df2, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))
df_combined = df1.merge(df2, left_on="GID_2", right_on="GDAM_id", how="inner")
df_combined.drop(["NAME_3", "NAME_4", "NAME_5"], axis=1, inplace=True)
print(f"*"*100)
print(tabulate(df_combined, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))

+---------+---------------+----------+----------+----------+----------+---------------+--------------+--------------+--------------+---------------+---------------+---------+-------------+---------+-------------+------------+--------------+---------+-------------+---------+-------------+---------+-------------+-------------+------------+----------+---------------+-------------+------------------+-------------------+------------------+-------------+--------------+---------------+----------+------------+------------+-----------------+--------------+-------------------+-----------------------------+-----------------------------+---------------------------------+--------------------+-------------------+-------------------+---------------------+-------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+--------------------+-------------------+------------------+------------------+--------------------+------------

In [12]:
# Combine geopandas dataframe with csv dataframe on GDAM_ID (see read_process_grid_data.py for the function that does this in the main code)
gpkg_path = Path("../data/input/DLL") / "data_final.gpkg"
csv_path = Path("../data/input/DLL") / "data_timeseries_70.csv"
merged_output_path_gpkg = Path("../data/processed/DLL") / "urban_classification_years.gpkg"
merged_output_path_parquet = Path("../data/processed/DLL") / "urban_classification_years.parquet"

# Only load the ID column plus geometry from the gpkg first, to check the join before pulling everything in
gdf_data_final = gpd.read_file(gpkg_path, layer="data_final")
df_data_timeseries = pd.read_csv(csv_path)

# print sample
print(gdf_data_final.sample(5))
print(f"*"*100)
print(df_data_timeseries.sample(5))

# Check join coverage before committing to a full merge
gpkg_ids = set(gdf_data_final["GDAM_ID"])
csv_ids = set(df_data_timeseries["GDAM_id"])

print("IDs in gpkg but not in csv:", len(gpkg_ids - csv_ids))
print("IDs in csv but not in gpkg:", len(csv_ids - gpkg_ids))

merged_gdf = gdf_data_final.merge(df_data_timeseries, left_on="GID_2", right_on="GDAM_id", how="left")
base_cols = ["UID", "GID_2", "NAME_1", "NAME_2", "geometry"]
cluster_cols = [col for col in merged_gdf.columns if col.startswith("cluster_")]
merged_gdf = merged_gdf[base_cols + cluster_cols]

# print
print(f"Number of features in merged GeoDataFrame: {len(merged_gdf)}")
print(merged_gdf.dtypes)
print(tabulate(merged_gdf.drop(columns=["geometry"]).sample(5), headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))

# save to .gpkg
print(f"Saving merged GeoDataFrame to gpkg file {merged_output_path_gpkg}...")
#merged_gdf.to_file(merged_output_path_gpkg, driver="GPKG")

# save to .parquet
print(f"Saving merged GeoDataFrame to parquet file {merged_output_path_parquet}...")
merged_gdf.to_parquet(merged_output_path_parquet)


           UID             NAME_1     NAME_2       NAME_3  NAME_4      NAME_5  \
306837  306838     Islas Baleares   Baleares    n.a. (59)   Petra        None   
284231  284232      Iburasirazuba  Nyagatare       Mukama  Gatete  Ryandahuka   
248205  248206  Negros Occidental     Murcia  Blumentritt    None        None   
192732  192733           Sardegna      Nuoro        Onani    None        None   
308807  308808               Bern    Aarberg     Kappelen    None        None   

               GGMCF         EDGAR  DEGURBA_L1  DEGURBA_L2  ...  \
306837  2.643701e+07  1.905313e+07         1.0        13.0  ...   
284231  2.071368e+03  2.990494e+04         1.0        12.0  ...   
248205  1.578755e+07  4.905648e+06         2.0        21.0  ...   
192732  1.745731e+06  3.853067e+06         1.0        12.0  ...   
308807  7.195451e+07  1.334522e+07         2.0        21.0  ...   

        POP_prop_adm_1  BUILT_prop_adm_1 Cluster_DDL      GGMCF_PC  \
306837        0.002534          0.006107

In [13]:
# check
missing_ids_from_csv = gpkg_ids - csv_ids
print("IDs in gpkg but not in csv:", missing_ids_from_csv)
print("-----------------------------------------------------")
missing_rows = gdf[gdf["GDAM_ID"].isin(missing_ids_from_csv)]
print(missing_rows[["GDAM_ID", "NAME_0", "NAME_1", "NAME_2", "CONTINENT"]])

IDs in gpkg but not in csv: {'CXR', 'XCA', 'GIB', 'FLK', 'NFK', 'MDV', 'PCN', 'VAT', 'MAF', 'GBR.1.6.1.3_1', 'CCK', 'BVT', 'XSP', 'GBR.3.27.1.3_1', 'MCO', 'SXM', 'XCL', 'ATA', 'GBR.1.6.1.14_1', 'NIU', 'GBR.1.6.1.20_1', 'ABW', 'HMD', 'XPI', 'SGS', 'KIR', 'IOT', 'CUW'}
-----------------------------------------------------
               GDAM_ID                            NAME_0    NAME_1  \
2856               ATA                        Antarctica      None   
3378               ABW                             Aruba      None   
19094              BVT                     Bouvet Island      None   
24667              IOT    British Indian Ocean Territory      None   
41684              XCA                       Caspian Sea      None   
44870              CXR                  Christmas Island      None   
44871              XCL                 Clipperton Island      None   
44872              CCK                     Cocos Islands      None   
47412              CUW                          

In [14]:
# print info merged data
print("Shape (rows, columns):", merged_gdf.shape)
print("Columns:", merged_gdf.columns.tolist())
print("Dtypes:")
print(merged_gdf.dtypes)
print("CRS:", merged_gdf.crs)
print("Geometry column:", merged_gdf.geometry.name)
print("Geometry type(s):", merged_gdf.geom_type.unique())
print("Memory usage (bytes, deep):")
print(merged_gdf.memory_usage(deep=True))
print("Total memory usage (MB):", merged_gdf.memory_usage(deep=True).sum() / 1e6)

print(merged_gdf.describe())
merged_gdf.info(memory_usage="deep")

Shape (rows, columns): (356508, 5)
Columns: ['UID', 'GID_2', 'NAME_1', 'NAME_2', 'geometry']
Dtypes:
UID            int64
GID_2         object
NAME_1        object
NAME_2        object
geometry    geometry
dtype: object
CRS: EPSG:4326
Geometry column: geometry
Geometry type(s): ['MultiPolygon']
Memory usage (bytes, deep):
Index            132
UID          2852064
GID_2       21002225
NAME_1      22804608
NAME_2      21868600
geometry     2852064
dtype: int64
Total memory usage (MB): 71.379693
                 UID
count  356508.000000
mean   178254.500000
std    102915.139222
min         1.000000
25%     89127.750000
50%    178254.500000
75%    267381.250000
max    356508.000000
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 356508 entries, 0 to 356507
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   UID       356508 non-null  int64   
 1   GID_2     356508 non-null  object  
 2   NAME_1    356485 non-null 